# Arbeitspaket (AP) 1: Management & Nutzung relationaler Daten

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Open Book, Eigener Computer, Internet-Zugang</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Nicht erlaubt ist der systematische Einsatz von generativer KI (z.B. Copilot, ChatGPT, Claude, Google's «Übersicht mit KI») für das Lösen der Aufgaben sowie beliebige Formen von Kommunikation oder Kollaboration mit anderen Menschen.</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung &amp; Design</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (cinema.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (cinema.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Antwort bzw. Ansatz im Kern falsch (z.B. falsches Konzept oder falsche Schlussfolgerung; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `cinema.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `movies`, `cinemas`, `staff`, `screenings`, `customers`, `tickets`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `cinema_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [1]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /workspaces/python_postgresql_pgadmin


### Datenbankverbindung zu PostgreSQL

In [6]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

OperationalError: (psycopg2.OperationalError) could not translate host name "None" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)

### Daten nach PostgreSQL laden

In [7]:
# Tabellen aus der SQLite-Datei cinema.db nach PostgreSQL übertragen
tables = ['movies', 'cinemas', 'staff', 'screenings', 'customers', 'tickets']

sqlite_engine = create_engine('sqlite:///cinema.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

OperationalError: (psycopg2.OperationalError) could not translate host name "None" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)

> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `cinema.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///cinema.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `cinema.db` werden nicht gewertet.*

### A1 (4 Punkte)

Die Tabelle `screenings` enthält `base_price` und die Tabelle `tickets` enthält `paid_price`.

Ein Datenbankarchitekt sagt: *«`paid_price` ist redundant — der Preis steht bereits in `screenings.base_price`. Die Spalte sollte entfernt werden.»*

Hat er recht? Begründen Sie Ihre Antwort mit einem konkreten Szenario aus dem Kinobetrieb (z.B. Rabattaktionen, Gutscheine).

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Das Datenteam analysiert die Tabelle `customers` und findet folgende Werte in der Spalte `membership_type`:

```
Standard, standard, STANDARD, Standart, Premium, premium, Student, student, Senior
```

Beschreiben Sie das konkrete Datenproblem und nennen Sie die passende Data-Quality-Eigenschaft aus den Vorlesungsfolien (z.B. accurate, consistent, complete, reliable).

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

In `cinema.db` ist `paid_price` bei keinem Ticket höher als der `base_price` der zugehörigen Vorstellung. Angenommen, ein neues Ticket hätte `paid_price` = CHF 18.00 bei einem `base_price` von CHF 16.00: Wäre das **zwingend** ein Datenfehler? Begründen Sie in einem Satz.

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `cinema.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl verkaufter Tickets pro Kino — für **alle** Kinos in der Datenbank.

```sql
SELECT c.name, COUNT(*) AS ticket_count
FROM screenings s
JOIN tickets t ON s.screeningid = t.screeningid
JOIN cinemas c ON s.cinemaid = c.cinemaid
GROUP BY c.name;
```

Die Tabelle `cinemas` enthält **6 Kinos**. Tatsächliches Ergebnis der SQL-Abfrage: nur **5 Zeilen**, Ticketanzahl zwischen 350 und 450.

Das Management fragt: *«Zeig mir auch Kinos, die noch keine Tickets verkauft haben.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 6 Kinos** erscheinen und ein Kino ohne Tickets korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

- FROM cinemas statt FROM screenings: Startet bei der Kino-Tabelle, damit alle 6 Kinos unabhängig von stattgefundenen Vorstellungen erfasst werden.
- LEFT JOIN Kaskade: Verbindet von cinemas ausgehend screenings und tickets, wodurch Kinos ohne Verkäufe im Datensatz erhalten bleiben.
- COUNT(t.ticketid) statt COUNT(*): COUNT(*) würde die leere LEFT JOIN-Zeile als 1 zählen. Die spezifische Spaltenzählung ignoriert NULL-Werte und gibt korrekt 0 aus.

**Korrigierte SQL-Abfrage:**

In [9]:
engine = create_engine('sqlite:///cinema.db')

df_B1 = pd.read_sql_query("""
                            SELECT
                                c.name,
                                COUNT(t.ticketid) AS ticket_count
                            FROM cinemas c
                            LEFT JOIN screenings s ON c.cinemaid = s.cinemaid
                            LEFT JOIN tickets t ON s.screeningid = t.screeningid
                            GROUP BY c.name;
                        """, con=engine)

engine.dispose()
df_B1

,name,ticket_count
0,Arthouse Metropol,354
1,Cineplexx,383
2,Kino Aurora,0
3,Kinok,441
4,Pathé Küchlin,436
5,Rex Kino,386


### B2 (8 Punkte)

**Ziel:** Liste aller Kunden (`customerid`, `firstname`, `lastname`), die noch nie ein Ticket gekauft haben.

```sql
SELECT c.customerid, c.firstname, c.lastname
FROM customers c
JOIN tickets t ON c.customerid = t.customerid
WHERE t.ticketid IS NULL;
```

Tatsächliches Ergebnis: 0 Zeilen — obwohl es Kunden ohne Tickets gibt.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, die alle Kunden ohne Ticket ausgibt. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
-- Korrigierte SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_B2

---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `cinema.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Die gültigen Werte in `customers.membership_type` sind `Standard`, `Premium`, `Student` und `Senior`. Schreiben Sie eine SQL-Abfrage, die alle Kunden (`customerid`, `firstname`, `lastname`, `membership_type`) mit einem ungültigen Wert in `membership_type` ausgibt — d.h. alle Werte ausser diesen vier. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C1

### C2 (4 Punkte)

Wie hoch ist der durchschnittliche bezahlte Ticketpreis (`paid_price`) pro Mitgliedschaftstyp (`membership_type`)? Geben Sie `membership_type` und den Durchschnittspreis (`avg_price`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_price`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C2

### C3 (4 Punkte)

Welche Kinos haben mehr als 400 Tickets verkauft? Geben Sie `name` und Anzahl Tickets (`ticket_count`) aus, absteigend sortiert nach `ticket_count`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C3

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')